In [ ]:
import os
import logging
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import urllib.parse # Thêm thư viện này để xử lý mật khẩu

# --- 1. CẤU HÌNH LOGGING ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler("qa_qc_eda_pipeline.log", encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

def run_eda_pipeline():
    logger.info("BẮT ĐẦU QUÁ TRÌNH KHÁM PHÁ VÀ KIỂM ĐỊNH DỮ LIỆU")

    # --- 2. KẾT NỐI DATABASE---
    load_dotenv()
    
    db_user = os.getenv("DB_USER")
    db_host = os.getenv("DB_HOST")
    db_port = os.getenv("DB_PORT")
    db_name = os.getenv("DB_NAME")
    raw_password = os.getenv("DB_PASSWORD")
    db_password = urllib.parse.quote_plus(raw_password) if raw_password else None

    # Kiểm tra xem có thiếu biến nào không
    if not all([db_user, db_password, db_host, db_port, db_name]):
        logger.error("Không tìm thấy đủ các biến môi trường (DB_USER, DB_PASSWORD, DB_HOST...) trong file .env!")
        return

    # Lắp ghép thành chuỗi kết nối PostgreSQL chuẩn
    DATABASE_URL = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"

    try:
        engine = create_engine(DATABASE_URL)
        logger.info("Đã kết nối thành công với Supabase.")
    except Exception as e:
        logger.critical(f"Lỗi kết nối database: {e}")
        return

    # --- 3. TRUY VẤN DỮ LIỆU TỪ BI MART ---
    # Đã sửa lại tên view cho chuẩn với schema của bạn
    query = "SELECT * FROM bi_mart.mv_bi_mart_hourly_measures"
    logger.info(f"Đang thực thi truy vấn kéo dữ liệu từ view: bi_mart.mv_bi_mart_hourly_measures")
    
    try:
        df = pd.read_sql_query(query, engine)
    except Exception as e:
        logger.error(f"Lỗi khi truy vấn dữ liệu: {e}")
        return

    # --- 4. KIỂM ĐỊNH & THỐNG KÊ (EDA) ---
    if df.empty:
        logger.error("View trả về 0 dòng! Dừng quy trình EDA. Cần kiểm tra lại luồng load từ DWH.")
        return
    
    logger.info(f"Đã kéo thành công {df.shape[0]} dòng và {df.shape[1]} cột.")

    # Kiểm tra Missing Values (QA/QC)
    missing_data = df.isnull().sum()
    total_missing = missing_data.sum()
    
    if total_missing == 0:
        logger.info("Chất lượng dữ liệu: Tốt. Không phát hiện dữ liệu khuyết thiếu.")
    else:
        logger.warning(f"Chất lượng dữ liệu: CẢNH BÁO. Phát hiện {total_missing} giá trị bị thiếu!")
        missing_cols = missing_data[missing_data > 0].to_dict()
        logger.warning(f"Chi tiết cột thiếu dữ liệu: {missing_cols}")

    # Thống kê mô tả toàn bộ dữ liệu
    logger.info("Đang tính toán các chỉ số thống kê mô tả cho toàn bộ dữ liệu...")
    df_numeric = df.select_dtypes(include=['number'])
    thong_ke_mo_ta = df_numeric.describe().T.round(2)
    
    report_path = "thong_ke_mo_ta_san_luong.csv"
    thong_ke_mo_ta.to_csv(report_path)
    logger.info(f"Đã xuất báo cáo thống kê mô tả ra file: {report_path}")

    # --- 5. THỐNG KÊ MÔ TẢ CHO SITE CÓ CAPACITY = NULL ---
    # Kiểm tra xem cột 'capacity' có tồn tại trong DataFrame không
    if 'capacity' in df.columns:
        logger.info("Đang lọc dữ liệu và tính toán thống kê mô tả cho các site có capacity = null...")
        
        # Lọc các dòng có giá trị capacity là null (NaN)
        df_capacity_null = df[df['capacity'].isnull()]
        
        if not df_capacity_null.empty:
            logger.info(f"Phát hiện {df_capacity_null.shape[0]} dòng có capacity = null.")
            
            # Chọn các cột số từ phần dữ liệu đã lọc
            df_null_numeric = df_capacity_null.select_dtypes(include=['number'])
            thong_ke_mo_ta_null = df_null_numeric.describe().T.round(2)
            
            report_null_path = "thong_ke_mo_ta_capacity_null.csv"
            thong_ke_mo_ta_null.to_csv(report_null_path)
            logger.info(f"Đã xuất báo cáo thống kê mô tả (capacity = null) ra file: {report_null_path}")
        else:
            logger.info("Không tìm thấy dòng nào có capacity = null trong dữ liệu.")
    else:
        logger.error("Không tìm thấy cột 'capacity' trong dữ liệu kéo về để thực hiện lọc!")

    logger.info("HOÀN TẤT.\n" + "-"*50)

if __name__ == "__main__":
    run_eda_pipeline()